In [1]:
from transformers import BitsAndBytesConfig

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # 使用NF4量化
    bnb_4bit_quant_type="nf4",
    # 是否使用双重量化
    bnb_4bit_use_double_quant=False,
    # 反量化之后的数据类型，会使用该类型参与到前向传播的计算
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-8B"  ,device_map="auto")

W0623 13:32:17.866000 46184 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 399/399 [00:12<00:00, 33.14it/s, Materializing param=model.norm.weight]                               
Some parameters are on the meta device because they were offloaded to the cpu.


In [5]:
torch.cuda.memory_allocated() / 1024 / 1024 /1024

8.347004890441895

In [6]:
from peft import get_peft_model,prepare_model_for_kbit_training
prepared_model = prepare_model_for_kbit_training(model)

In [9]:
from peft import LoraConfig
lora_config = LoraConfig(
    r=4,
    lora_alpha=4,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM"
)
quantized_peft_model = get_peft_model(prepared_model,lora_config)

In [25]:
from datasets import load_dataset

# 1、处理数据，处理成type为language modeling ，format为对话格式的数据
psychology_data = load_dataset("json",data_files={"train":r"./data/psychology_data.jsonl"})
selected_psychology_data = psychology_data["train"].select(range(100))
selected_psychology_data

Dataset({
    features: ['conversation_id', 'category', 'conversation', 'dataset'],
    num_rows: 100
})

In [11]:
psychology_data["train"].select

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 77250
    })
})

In [21]:
splited_data =psychology_data.train_test_split(test_size=0.2,train_size=0.8)

AttributeError: 'DatasetDict' object has no attribute 'train_test_split'

In [20]:
splited_data

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 61800
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 15450
    })
})